# 09. Causal Forests

## Background

All methods so far estimate a single ATE or ATT — an average over a population. But treatment effects are rarely homogeneous. A job training program may help workers in their 30s a lot and do nothing for workers near retirement. A drug may be effective for one genetic profile and harmful for another. **Heterogeneous Treatment Effects (HTE)** estimation is the problem of learning τ(x) = E[Y(1) − Y(0) | X=x].

Causal forests (Wager & Athey 2018) extend random forests to estimate CATE (Conditional Average Treatment Effect) τ(x). The key insight: a regression forest predicts outcomes by finding "similar" training examples. A causal forest finds examples that are similar in their *treatment effects* — it targets the causal contrast rather than the level.

## What You'll Learn

- CATE estimation: τ(x) = E[Y(1)−Y(0)|X=x]
- Honest splitting: using separate subsamples for tree building and leaf estimation
- The GRF (Generalized Random Forest) algorithm: solving a local moment equation
- EconML's causal forest: practical implementation with confidence intervals
- CATE visualization and calibration
- Policy learning: who should we treat given heterogeneous effects?

## Why This Matters

Causal forests from the `econml` package (Microsoft Research) are now standard for HTE analysis in tech industry experimentation. They produce individual-level treatment effect estimates with valid confidence intervals (via the `predict` + `predict_interval` API). This enables policy questions: "given a budget to treat 30% of users, which 30% maximizes total benefit?" — an uplift modeling problem (notebook 11).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings; warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
def simulate_hte(n=3000, p=10, seed=0):
    """
    DGP with heterogeneous treatment effects.
    True CATE: τ(x) = 2 + x0 + sin(x1) — varies substantially across units.
    Confounded: P(T=1|X) depends on X.
    """
    rng = np.random.default_rng(seed)
    X = rng.normal(0, 1, (n, p))

    # True CATE function — heterogeneous
    tau_x = 2 + X[:, 0] + np.sin(X[:, 1])

    # Confounded propensity
    ps = 1 / (1 + np.exp(-0.5 * X[:, 0] + 0.3 * X[:, 1]))
    T = rng.binomial(1, ps)

    y0 = X[:, 0]**2 + X[:, 1] + rng.normal(0, 1, n)
    y1 = y0 + tau_x
    Y = np.where(T, y1, y0)

    return X, T, Y, tau_x, ps


X, T, Y, tau_true, ps = simulate_hte(n=3000, p=10)
print(f"True ATE  = {tau_true.mean():.3f}")
print(f"True CATE range: [{tau_true.min():.2f}, {tau_true.max():.2f}]")
print(f"CATE std  = {tau_true.std():.3f}  (substantial heterogeneity)")
print(f"Naïve DiM = {Y[T==1].mean()-Y[T==0].mean():.3f}  (biased)")

## 1. Why Regression Forests Fail at CATE

A naive approach: fit a forest on (T, X) → Y, then compute τ̂(x) = forest(1, x) − forest(0, x). This fails because forests minimize MSE for Y — they don't target the difference. The bias from the outcome level bleeds into the contrast.

In [ ]:
# Naive S-learner (single model with T as feature)
from sklearn.ensemble import RandomForestRegressor

Xt = np.column_stack([T.reshape(-1,1), X])
X_train, X_test, T_train, T_test, Y_train, Y_test, tau_train, tau_test =     train_test_split(Xt, T, Y, tau_true, test_size=0.3, random_state=0)

rf_s = RandomForestRegressor(n_estimators=200, max_depth=6, min_samples_leaf=20, random_state=0)
rf_s.fit(X_train, Y_train)

# Predict τ(x) by difference
X_test_1 = X_test.copy(); X_test_1[:,0] = 1
X_test_0 = X_test.copy(); X_test_0[:,0] = 0
tau_hat_s = rf_s.predict(X_test_1) - rf_s.predict(X_test_0)

rmse_s = np.sqrt(np.mean((tau_hat_s - tau_test)**2))
print(f"S-learner RMSE: {rmse_s:.3f}")

# T-learner: separate forests per arm
X_pure = X  # without T
X_tr, X_te, T_tr, T_te, Y_tr, Y_te, tau_tr, tau_te =     train_test_split(X_pure, T, Y, tau_true, test_size=0.3, random_state=0)

rf_1 = RandomForestRegressor(n_estimators=200, max_depth=6, min_samples_leaf=20, random_state=0)
rf_0 = RandomForestRegressor(n_estimators=200, max_depth=6, min_samples_leaf=20, random_state=0)
rf_1.fit(X_tr[T_tr==1], Y_tr[T_tr==1])
rf_0.fit(X_tr[T_tr==0], Y_tr[T_tr==0])
tau_hat_t = rf_1.predict(X_te) - rf_0.predict(X_te)
rmse_t = np.sqrt(np.mean((tau_hat_t - tau_te)**2))
print(f"T-learner RMSE: {rmse_t:.3f}")
print(f"(Naïve learners work but don't have valid CIs)")

## 2. EconML Causal Forest

The `econml` package implements GRF-based causal forests with:
- Double ML residualization for deconfounding
- Honest splitting for valid inference
- Confidence intervals via the infinitesimal jackknife

In [ ]:
try:
    from econml.grf import CausalForest
    from econml.dml import CausalForestDML

    # CausalForestDML: double ML + causal forest for CATE with confounders
    cf = CausalForestDML(
        model_y=GradientBoostingRegressor(n_estimators=100, max_depth=3),
        model_t=GradientBoostingRegressor(n_estimators=100, max_depth=3),
        n_estimators=200,
        min_samples_leaf=20,
        max_depth=None,
        random_state=42,
        cv=5,
    )
    cf.fit(Y_tr, T_tr, X=X_tr)

    tau_hat_cf = cf.effect(X_te)
    tau_ci     = cf.effect_interval(X_te, alpha=0.05)

    rmse_cf = np.sqrt(np.mean((tau_hat_cf - tau_te)**2))
    print(f"CausalForestDML RMSE: {rmse_cf:.3f}  (vs S-learner {rmse_s:.3f})")

    # CATE heterogeneity test
    print("\nRate test (heterogeneity vs constant effect):")
    rate = cf.score(Y_te, T_te, X=X_te)
    print(f"  Score: {rate:.4f}")

    ECONML_AVAILABLE = True

except ImportError:
    print("econml not installed — using DR-learner approximation")
    ECONML_AVAILABLE = False

In [ ]:
# DR-learner: doubly robust CATE via pseudo-outcomes (works without econml)
from sklearn.model_selection import cross_val_predict

def dr_learner(Y, T, X, n_folds=5, seed=0):
    """
    Doubly Robust (DR) learner for CATE.
    Construct pseudo-outcome Γ_i, then regress on X.
    Γ_i = μ̂₁(X) - μ̂₀(X) + T(Y - μ̂₁(X))/ê(X) - (1-T)(Y - μ̂₀(X))/(1-ê(X))
    Γ_i is an unbiased pointwise estimate of τ(X_i).
    """
    from sklearn.model_selection import KFold
    n = len(Y)
    mu1_hat = np.zeros(n); mu0_hat = np.zeros(n); ps_hat = np.zeros(n)

    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(X):
        m1 = GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=seed)
        m0 = GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=seed)
        mp = GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=seed)

        X_tr, X_vl = X[tr_idx], X[val_idx]
        T_tr_fold = T[tr_idx]

        m1.fit(X_tr[T_tr_fold==1], Y[tr_idx][T_tr_fold==1])
        m0.fit(X_tr[T_tr_fold==0], Y[tr_idx][T_tr_fold==0])
        mp.fit(X_tr, T_tr_fold.astype(float))

        mu1_hat[val_idx] = m1.predict(X_vl)
        mu0_hat[val_idx] = m0.predict(X_vl)
        ps_hat[val_idx]  = np.clip(mp.predict(X_vl), 0.05, 0.95)

    # DR pseudo-outcome
    gamma = (mu1_hat - mu0_hat
             + T * (Y - mu1_hat) / ps_hat
             - (1-T) * (Y - mu0_hat) / (1 - ps_hat))

    # Regress pseudo-outcome on X
    rf_cate = RandomForestRegressor(n_estimators=300, max_depth=6, min_samples_leaf=15, random_state=0)
    rf_cate.fit(X, gamma)
    return rf_cate, gamma


rf_dr, gamma = dr_learner(Y_tr, T_tr, X_tr)
tau_hat_dr = rf_dr.predict(X_te)
rmse_dr = np.sqrt(np.mean((tau_hat_dr - tau_te)**2))
print(f"DR-learner RMSE: {rmse_dr:.3f}")

In [ ]:
# Visualize CATE estimates vs truth along x0
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sort_idx = np.argsort(X_te[:, 0])
x0_sorted = X_te[sort_idx, 0]
tau_true_sorted = tau_te[sort_idx]
tau_dr_sorted   = tau_hat_dr[sort_idx]

axes[0].scatter(x0_sorted, tau_true_sorted, alpha=0.2, s=8, color='gray', label='True τ(x)')
axes[0].scatter(x0_sorted, tau_dr_sorted,   alpha=0.2, s=8, color='#F44336', label='DR-learner τ̂(x)')
axes[0].set_xlabel('X[0] (strongest CATE modifier)')
axes[0].set_ylabel('CATE')
axes[0].set_title('CATE Estimates vs Truth along X[0]')
axes[0].legend()

# CATE distribution
axes[1].hist(tau_hat_dr, bins=50, alpha=0.7, color='#F44336', label=f'Estimated CATE (std={tau_hat_dr.std():.2f})', density=True)
axes[1].hist(tau_te, bins=50, alpha=0.5, color='gray', label=f'True CATE (std={tau_te.std():.2f})', density=True)
axes[1].axvline(tau_te.mean(), color='black', lw=2, label=f'ATE = {tau_te.mean():.2f}')
axes[1].set_xlabel('CATE'); axes[1].set_ylabel('Density')
axes[1].set_title('CATE Distribution — Heterogeneous Effects Captured')
axes[1].legend()

plt.tight_layout()
plt.savefig('09_cate.png', dpi=110, bbox_inches='tight')
plt.show()

## 3. Policy Learning from CATE Estimates

Given CATE estimates, build a targeting rule: treat unit i iff τ̂(x_i) > cost_threshold. This converts CATE estimates into a treatment allocation policy.

In [ ]:
def policy_value(tau_true, tau_hat, cost_per_treatment=1.5):
    """
    Compare policies:
    - Treat all: value = E[τ] - cost
    - Treat none: value = 0
    - Target by τ̂: treat iff τ̂(x) > cost
    - Oracle: treat iff τ(x) > cost  (infeasible upper bound)
    """
    treat_all    = tau_true.mean() - cost_per_treatment
    treat_none   = 0.0
    oracle       = np.mean(np.maximum(0, tau_true - cost_per_treatment))
    targeted     = np.mean(np.where(tau_hat > cost_per_treatment,
                                    tau_true - cost_per_treatment, 0))
    return dict(treat_all=treat_all, treat_none=treat_none,
                oracle=oracle, targeted=targeted,
                targeted_fraction=np.mean(tau_hat > cost_per_treatment))


pv = policy_value(tau_te, tau_hat_dr, cost_per_treatment=2.0)
print(f"Policy comparison (cost per treatment = 2.0):")
print(f"  Treat all      : {pv['treat_all']:.4f}")
print(f"  Treat none     : {pv['treat_none']:.4f}")
print(f"  Oracle policy  : {pv['oracle']:.4f}  (infeasible upper bound)")
print(f"  Targeted (DR)  : {pv['targeted']:.4f}  ({pv['targeted_fraction']:.1%} treated)")

# Policy value across different cost thresholds
costs = np.linspace(0.5, 5.0, 30)
pv_oracle = [np.mean(np.maximum(0, tau_te - c)) for c in costs]
pv_targeted = [np.mean(np.where(tau_hat_dr > c, tau_te - c, 0)) for c in costs]
pv_all = [tau_te.mean() - c for c in costs]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(costs, pv_oracle,   'k--', lw=2, label='Oracle policy')
ax.plot(costs, pv_targeted, 'r-',  lw=2, label='CATE-targeted policy (DR-learner)')
ax.plot(costs, pv_all,      'b:',  lw=2, label='Treat all')
ax.axhline(0, color='gray', lw=1)
ax.set_xlabel('Cost per treatment'); ax.set_ylabel('Expected policy value')
ax.set_title('Policy Value vs Treatment Cost')
ax.legend()
plt.tight_layout()
plt.savefig('09_policy_value.png', dpi=110, bbox_inches='tight')
plt.show()